In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd


def find_project_root(current_path, marker="config.yaml"):
    current = Path(current_path).resolve()
    for _ in range(5):  # Ищем на 5 уровней вверх
        if (current / marker).exists():
            return current
        current = current.parent
    raise FileNotFoundError(f"Не удалось найти корень проекта (файл {marker})")


PROJECT_ROOT = find_project_root(os.getcwd())
sys.path.append(str(PROJECT_ROOT))
print(f"Корневая директория: {PROJECT_ROOT}")


Корневая директория: D:\Education\Arcticle\02_dtp_project_JAER


In [2]:
from etl.utils import load_config

config = load_config(PROJECT_ROOT / "config.yaml")
DATA_PATH = PROJECT_ROOT / config['paths']['full_df']
MODELS_DIR = PROJECT_ROOT / config['paths']['models']

os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Данные берем из: {DATA_PATH}")
print(f"Модели сохраняем в: {MODELS_DIR}")

Данные берем из: D:\Education\Arcticle\02_dtp_project_JAER\data\processed\dtp_full_dataset.parquet
Модели сохраняем в: D:\Education\Arcticle\02_dtp_project_JAER\data\models


In [3]:
df = pd.read_parquet(DATA_PATH)
df.shape

(1465882, 89)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167449 entries, 0 to 167448
Data columns (total 89 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   light_cat                       167449 non-null  object 
 1   scheme                          167449 non-null  object 
 2   category                        167449 non-null  object 
 3   year                            167449 non-null  int32  
 4   month_sin                       167449 non-null  float64
 5   month_cos                       167449 non-null  float64
 6   hour_sin                        167449 non-null  float64
 7   hour_cos                        167449 non-null  float64
 8   is_weekend                      167449 non-null  int64  
 9   near_residential                167449 non-null  int64  
 10  near_pedestrian_infrastructure  167449 non-null  int64  
 11  near_road_junctions             167449 non-null  int64  
 12  near_education  

In [6]:
print("=== СТРУКТУРА ДАТАСЕТА ===")

prefixes = {
    "TARGET": ["target"],
    "GEO": ["lat", "long", "region_id"],
    "TIME": ["datetime", "year", "month_", "hour_", "is_weekend"],
    "LIGHT": ["light_"],
    "NEARBY": ["near_", "nearby_"],
    "WEATHER": ["weather_"],
    "ROAD": ["road_"],
    "VEHICLES (Железо)": ["vh_"],
    "PEOPLE (Люди)": ["ppl_"],
    "VIOLATIONS (Нарушения)": ["viol_"],
    "OTHER (Категории и пр.)": ["category", "scheme"]
}

found_cols = set()

for group, tags in prefixes.items():
    cols = [c for c in df.columns if any(c.startswith(t) for t in tags)]
    if cols:
        print(f"\n{group} ({len(cols)} шт.):")
        print("\t", ", ".join(cols))
        found_cols.update(cols)

rest = list(set(df.columns) - found_cols)
if rest:
    print(f"\nОСТАЛЬНЫЕ ({len(rest)} шт.):")
    print(", ".join(rest))

=== СТРУКТУРА ДАТАСЕТА ===

TARGET (1 шт.):
	 target

GEO (3 шт.):
	 lat, long, region_id

TIME (6 шт.):
	 year, month_sin, month_cos, hour_sin, hour_cos, is_weekend

LIGHT (1 шт.):
	 light_cat

NEARBY (13 шт.):
	 near_residential, near_pedestrian_infrastructure, near_road_junctions, near_education, near_commercial, near_public_transport, near_railway, near_transport_hubs, near_administrative, near_healthcare, near_road_objects, near_other, nearby_objects_count

WEATHER (8 шт.):
	 weather_clear, weather_fog, weather_hot, weather_overcast, weather_rain, weather_snow, weather_wind, weather_other

ROAD (16 шт.):
	 road_bad_fences, road_bad_infrastructure, road_bad_lights, road_bad_maintenance, road_bad_markings, road_bad_signs, road_bad_surface, road_dirty, road_dry, road_ice, road_no_light, road_obstacles, road_roadworks, road_snow, road_wet, road_other

VEHICLES (Железо) (20 шт.):
	 vh_count, vh_is_solo, vh_is_mass, vh_heavy_light_conflict, vh_age_gap, vh_max_age, vh_mean_age, vh_count_

In [5]:
nans = df.isna().sum()
nans = nans[nans > 0]
if not nans.empty:
    print("НАЙДЕНЫ ПРОПУСКИ (NaN):")
    print(nans)
else:
    print("Пропусков нет. Датасет плотный.")

Пропусков нет. Датасет плотный.


In [6]:
print("ТИПЫ ДАННЫХ:")
print(df.dtypes.value_counts())

ТИПЫ ДАННЫХ:
int64      71
float64    11
object      4
int32       3
Name: count, dtype: int64


In [7]:
print("=== РАСПРЕДЕЛЕНИЕ ТАРГЕТА ===")
target_counts = df['target'].value_counts().sort_index()
print(target_counts)
print("\nВ процентах:")
print(df['target'].value_counts(normalize=True).sort_index().mul(100).round(2).astype(str) + '%')

=== РАСПРЕДЕЛЕНИЕ ТАРГЕТА ===
target
0    829755
1    495775
2    140352
Name: count, dtype: int64

В процентах:
target
0     56.6%
1    33.82%
2     9.57%
Name: proportion, dtype: object


In [8]:
display(df.sample(3))

,light_cat,scheme,category,year,month_sin,month_cos,hour_sin,hour_cos,is_weekend,near_residential,...,ppl_drv_novice,viol_priority,viol_pedestrian,viol_speed,viol_drunk,viol_safety,viol_rights,viol_runaway,viol_admin,target
1316748,night_dark,430,столкновение,2021,0.866025,5.000000e-01,-0.965926,0.258819,0,1,...,0,1,0,1,0,0,0,0,0,0
1390808,night_lit,740,наезд_на_пешехода,2022,-0.500000,8.660254e-01,-0.965926,-0.258819,0,0,...,0,0,1,0,0,0,0,1,0,0
26098,day,500,столкновение,2021,-1.000000,-1.836970e-16,-0.866025,0.500000,0,1,...,0,1,0,0,0,0,0,0,0,0


In [9]:
# 1) Виды ТС должны быть <= общего числа ТС
veh_types_sum = df[['vh_count_car','vh_count_truck','vh_count_bus','vh_count_moto','vh_count_special']].sum(axis=1)
print("veh_types_sum > vh_count:", (veh_types_sum > df['vh_count']).sum())

# 2) Группы брендов обычно <= vh_count (если бывают None бренды)
brand_sum = df[['vh_count_brand_ru','vh_count_brand_premium','vh_count_brand_chinese',
                'vh_count_brand_mass','vh_count_brand_commercial']].sum(axis=1)
print("brand_sum > vh_count:", (brand_sum > df['vh_count']).sum())

# 3) Цвета <= vh_count (если цвет бывает None/не заполнено)
color_sum = df[['vh_count_color_dark','vh_count_color_light','vh_count_color_colored']].sum(axis=1)
print("color_sum > vh_count:", (color_sum > df['vh_count']).sum())

veh_types_sum > vh_count: 0
brand_sum > vh_count: 0
color_sum > vh_count: 0


In [10]:
df['region_id'].value_counts()

region_id
омская_область_омск                                            21190
республика_татарстан_татарстан_казань                          17915
челябинская_область_челябинск                                  17189
республика_башкортостан_уфа                                    17175
тюменская_область_тюмень                                       16566
                                                               ...  
республика_саха_якутия_эвенобытантайский_национальный_район        1
мурманская_область_островной                                       1
свердловская_область_мо_пос_уральский                              1
севастополь_инкерман                                               1
архангельская_область_новая_земля                                  1
Name: count, Length: 2237, dtype: int64

In [13]:
df['category'].value_counts()

category
столкновение              65693
наезд_на_пешехода         50032
съезд_с_дороги            11961
наезд_на_препятствие      10111
опрокидывание              7628
наезд_на_велосипедиста     7611
падение_пассажира          6544
наезд_на_стоящее_тс        5994
иной_вид_дтп               1616
наезд_на_рабочего           259
Name: count, dtype: int64

In [14]:
df['vh_count_brand_mass'].value_counts()

vh_count_brand_mass
1     77237
0     64253
2     22499
3      2709
4       610
5        91
6        33
7        10
8         4
9         2
33        1
Name: count, dtype: int64

In [16]:
df['viol_speed'].value_counts()

viol_speed
0    124764
1     42685
Name: count, dtype: int64